In [0]:
import dlt
from pyspark.sql.functions import col, trim, when, avg, count

csv_source_path = "/Workspace/Users/nurfadisyah@gmail.com/" 

@dlt.table(
    name="churn_bronze",
    comment="Data mentah churn dari file CSV."
)
def churn_bronze():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("pathGlobFilter", "*.csv") 
        .load(csv_source_path)
    )

@dlt.table(
    name="churn_silver",
    comment="Data churn bersih, siap untuk ML (tipe data diperbaiki)."
)
@dlt.expect_or_drop("customerID_valid", "customerID IS NOT NULL")
def churn_silver():
    
    # Tentukan kolom numerik yang perlu diisi jika null
    numeric_cols_to_fill = ["tenure", "MonthlyCharges", "SeniorCitizen", "TotalCharges"]
    
    base_df = (
        dlt.read_stream("churn_bronze")
        .withColumn("tenure", col("tenure").cast("integer"))
        .withColumn("MonthlyCharges", col("MonthlyCharges").cast("double"))
        .withColumn("SeniorCitizen", col("SeniorCitizen").cast("integer"))
        .withColumn(
            "TotalCharges",
            when(trim(col("TotalCharges")) == '', None) # Ubah spasi ' ' menjadi null
            .otherwise(col("TotalCharges").cast("double"))
        )
        .withColumn(
            "Churn",
            when(col("Churn") == "Yes", 1).otherwise(0)
        )
    )
    
    # isi nilai null dengan 0
    return base_df.na.fill(0, subset=numeric_cols_to_fill)

@dlt.table(
    name="churn_gold_by_contract",
    comment="Data agregat: tingkat churn berdasarkan tipe kontrak."
)
def churn_gold_by_contract():
    return (
        dlt.read("churn_silver")
        .groupBy("Contract")
        .agg(
            avg("MonthlyCharges").alias("Rata2_Biaya_Bulanan"),
            avg("Churn").alias("Tingkat_Churn"),
            count("*").alias("Jumlah_Pelanggan")
        )
    )